# 16.12 - Consolidate provisional visual review

Combine the inspected source-train examples from 16.10, 16.11, 16.13 and 16.14 and the explicit crowding review batch into one auditable research manifest. Preserve uncertain target states and exact source provenance. This notebook does not create a training dataset or claim clinical ground truth.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display
root=Path.cwd().resolve()
while not (root / "AGENTS.md").exists():
    if root==root.parent: raise RuntimeError("Project root not found")
    root=root.parent
base=root / "data/processed/alignment_omni_frontal_candidates"
evidence=root / "docs/reports/alignment/agent_visual_pilot"
source=pd.read_csv(base / "review_queue.csv",dtype={"source_image_id":str,"sha256":str}).set_index("source_image_id")
exclude=set(pd.read_csv(root / "docs/reports/alignment/omni_frontal_candidate_audit/conservative_train_exclusions.csv",dtype=str).source_image_id)

## 1. Keep both target statuses explicit

For earlier positive boxes, the other target remains uncertain. The later batch contains one image explicitly inspected as negative for both targets. No silent negative labels are introduced.

In [2]:
first=pd.read_csv(evidence / "agent_visual_proposals.csv",dtype={"source_image_id":str})
second=pd.read_csv(evidence / "active_review_states.csv",dtype={"source_image_id":str})
third=pd.read_csv(evidence / "heuristic_review_states.csv",dtype={"source_image_id":str})
records=[]
for r in first.itertuples():
    records.append({"source_image_id":r.source_image_id,"view_review":"frontal",
        "diastema_review":"present" if r.target=="upper_midline_diastema" else "uncertain",
        "crowding_review":"present" if r.target=="anterior_crowding" else "uncertain",
        "reviewer":r.reviewer,"review_state":r.review_state,"visual_reason":r.visual_reason})
records += second[["source_image_id","view_review","diastema_review","crowding_review","reviewer","review_state","visual_reason"]].to_dict("records")
records += third[["source_image_id","view_review","diastema_review","crowding_review","reviewer","review_state","visual_reason"]].to_dict("records")
fourth=pd.read_csv(evidence / "negative_review_states.csv",dtype={"source_image_id":str})
records += fourth[["source_image_id","view_review","diastema_review","crowding_review","reviewer","review_state","visual_reason"]].to_dict("records")
fifth=pd.read_csv(evidence / "crowding_review_states.csv",dtype={"source_image_id":str})
records += fifth[["source_image_id","view_review","diastema_review","crowding_review","reviewer","review_state","visual_reason"]].to_dict("records")
manifest=pd.DataFrame(records)
assert not manifest.source_image_id.duplicated().any()
for r in manifest.itertuples():
    s=source.loc[r.source_image_id]
    assert s["split"]=="train" and r.source_image_id not in exclude
    assert hashlib.sha256((root / s["image_path"]).read_bytes()).hexdigest()==s["sha256"]
manifest.insert(1,"source_split","train")
manifest.insert(2,"source_sha256",[source.loc[id,"sha256"] for id in manifest.source_image_id])
display(manifest)

,source_image_id,source_split,source_sha256,view_review,diastema_review,crowding_review,reviewer,review_state,visual_reason
0,201916202112081,train,30b3c48e4df0bd79192687457148358c3f4403171384a2...,frontal,present,uncertain,Codex visual review,agent_visual_provisional,visible central separation
1,202139202205251,train,0c4791cf185aa50e57d7305704f0e26858793e2df99d61...,frontal,present,uncertain,Codex visual review,agent_visual_provisional,visible central separation
2,202212202202071,train,bb85ae53e7e0a6d14e0657a487f38cd853df01a37dd6ee...,frontal,present,uncertain,Codex visual review,agent_visual_provisional,visible central separation
3,202362202308141,train,b9e6d97fb002818f2520b45d012dcf9c0ff1ce4a93c91f...,frontal,present,uncertain,Codex visual review,agent_visual_provisional,visible central separation
4,202244202207131,train,ca4fa32d6ff49cbd45b8ecfba2b3618bfa278c173890fa...,frontal,uncertain,present,Codex visual review,agent_visual_provisional,visible anterior displacement and overlap
5,201875201810241,train,c1c153560b6d84dba654e57a2fb37f15dde520b612a511...,frontal,present,uncertain,Codex visual review,agent_visual_provisional,visible upper central separation; zero-score m...
6,202243202207131,train,9d53c6092befb3e88f75d56c0021a3f5a6e5f050f1fc60...,frontal,present,uncertain,Codex visual review,agent_visual_provisional,visible upper central separation
7,201883201809131,train,83a7de6470caf9db188484d6de4e108658ec6d193e614e...,frontal,absent,absent,Codex visual review,agent_visual_provisional,upper central contact and no obvious anterior ...
8,202211202201261,train,413f35c881649a2d0a4355d3f0dc0e41245b70fd456776...,frontal,present,uncertain,Codex visual review,agent_visual_provisional,visible upper central separation
9,202242202207211,train,8a35b49e9a3507f8ce728a2ba9b98796e815971e69bbb7...,frontal,present,uncertain,Codex visual review,agent_visual_provisional,visible upper central separation


## 2. Check target boxes against positive statuses

Each present target needs at least one box. Absent and uncertain statuses must have no box of that target. Validate original-image coordinates.

In [3]:
extra=pd.read_csv(evidence / "active_review_boxes.csv",dtype={"source_image_id":str})
third_boxes=pd.read_csv(evidence / "heuristic_review_boxes.csv",dtype={"source_image_id":str})
fourth_boxes=pd.read_csv(evidence / "crowding_review_boxes.csv",dtype={"source_image_id":str})
boxes=pd.concat([first[["source_image_id","target","x_min","y_min","x_max","y_max","reviewer","review_state"]],extra,third_boxes,fourth_boxes],ignore_index=True)
targets={"upper_midline_diastema":"diastema_review","anterior_crowding":"crowding_review"}
for r in manifest.itertuples():
    for target,field in targets.items():
        count=len(boxes[(boxes.source_image_id==r.source_image_id)&(boxes.target==target)])
        assert (count>0)==(getattr(r,field)=="present"),f"Target/box mismatch: {r.source_image_id} {target}"
for r in boxes.itertuples():
    s=source.loc[r.source_image_id]
    assert 0<=r.x_min<r.x_max<=int(s["width"]) and 0<=r.y_min<r.y_max<=int(s["height"])
print("Images:",len(manifest),"| Boxes:",len(boxes))
display(pd.DataFrame({field:manifest[field].value_counts() for field in ("diastema_review","crowding_review")}))

Images: 22 | Boxes: 16


,diastema_review,crowding_review
absent,6,3
present,14,2
uncertain,2,17


## 3. Save the provisional versioned snapshot

The versioned snapshot is intentionally separate from the original OMNI annotations and approved review queue. It is a small teaching/evidence batch, not yet a valid model training split.

In [4]:
for name,frame in (("agent_visual_manifest_v4.csv",manifest),("agent_visual_boxes_v4.csv",boxes)):
    path=evidence / name
    frame.to_csv(path,index=False)
    print("Saved",path)
print("Next: expand reviewed positives and negatives, create independent reviewed validation support, resolve overlap and rights before training.")

Saved C:\Users\huynh\Desktop\oral-disease\docs\reports\alignment\agent_visual_pilot\agent_visual_manifest_v4.csv
Saved C:\Users\huynh\Desktop\oral-disease\docs\reports\alignment\agent_visual_pilot\agent_visual_boxes_v4.csv
Next: expand reviewed positives and negatives, create independent reviewed validation support, resolve overlap and rights before training.
